In [1]:
import os
os.environ["PYTHONNET_RUNTIME"] = "coreclr"
os.environ["DOTNET_SYSTEM_DRAWING_USE_GDIPLUS"] = "1"

import clr
import numpy as np
import time
from datetime import timedelta
from pythonnet import load
from System import Environment, Array, String

# Load .NET Core runtime
load("coreclr")

# Try to import System.IO; fallback to os.chdir if it fails
try:
    from System.IO import Directory, Path, File
    use_dotnet_dir = True
except Exception as e:
    print(f"Note: System.IO import failed ({e}), using os.chdir instead")
    import os
    use_dotnet_dir = False

# Define DWSIM path
dwsimpath = "/usr/local/lib/dwsim/"

# Set working directory
if use_dotnet_dir:
    Directory.SetCurrentDirectory(dwsimpath)
else:
    os.chdir(dwsimpath)

# Add DWSIM assemblies
clr.AddReference(dwsimpath + "CapeOpen.dll")
clr.AddReference(dwsimpath + "DWSIM.Automation.dll")
clr.AddReference(dwsimpath + "DWSIM.Interfaces.dll")
clr.AddReference(dwsimpath + "DWSIM.GlobalSettings.dll")
clr.AddReference(dwsimpath + "DWSIM.SharedClasses.dll")
clr.AddReference(dwsimpath + "DWSIM.Thermodynamics.dll")
clr.AddReference(dwsimpath + "DWSIM.UnitOperations.dll")
clr.AddReference(dwsimpath + "DWSIM.Inspector.dll")
clr.AddReference(dwsimpath + "System.Buffers.dll")
clr.AddReference(dwsimpath + "DWSIM.Thermodynamics.ThermoC.dll")

# Now import DWSIM types
from DWSIM.Interfaces.Enums.GraphicObjects import ObjectType
from DWSIM.Thermodynamics import Streams, PropertyPackages
from DWSIM.UnitOperations import UnitOperations
from DWSIM.Automation import Automation3
from DWSIM.GlobalSettings import Settings

print("DWSIM imports successful!")

DWSIM imports successful!


In [2]:
# Create an instance of the Automation3 class from the DWSIM.Automation module
# This class provides methods for automating tasks in DWSIM, such as creating and manipulating flowsheets
interf = Automation3()

In [3]:
# Creates a flowsheet
sim = interf.CreateFlowsheet()

In [4]:
# Add Compounds
sim.AddCompound("Methane")
sim.AddCompound("Oxygen")
sim.AddCompound("CarbonDioxide")
sim.AddCompound("Water")
sim.AddCompound("Nitrogen")

In [5]:
fuel = sim.AddFlowsheetObject('Material Stream','fuel')
air = sim.AddFlowsheetObject('Material Stream','air')
feed = sim.AddFlowsheetObject('Material Stream','feed')
flue_gas = sim.AddFlowsheetObject('Material Stream','flue_gas')
residue = sim.AddFlowsheetObject('Material Stream', 'residue')
air_fuel_mixer = sim.AddFlowsheetObject('Stream Mixer','air_fuel_mixer')
combustor = sim.AddFlowsheetObject('Conversion Reactor','combustor')

In [6]:
fuel = fuel.GetAsObject()
air = air.GetAsObject()
feed = feed.GetAsObject()
flue_gas = flue_gas.GetAsObject()
residue = residue.GetAsObject()
air_fuel_mixer = air_fuel_mixer.GetAsObject()
combustor = combustor.GetAsObject()

In [7]:
sim.ConnectObjects(fuel.GraphicObject, air_fuel_mixer.GraphicObject, -1, -1)
sim.ConnectObjects(air.GraphicObject, air_fuel_mixer.GraphicObject, -1, -1)
sim.ConnectObjects(air_fuel_mixer.GraphicObject, feed.GraphicObject, -1, -1)
sim.ConnectObjects(feed.GraphicObject, combustor.GraphicObject, -1, -1)
sim.ConnectObjects(combustor.GraphicObject, flue_gas.GraphicObject, -1, -1)
sim.ConnectObjects(combustor.GraphicObject, residue.GraphicObject, -1, -1)

In [8]:
sim.AutoLayout()

In [9]:
fuel.SetOverallComposition(Array[float]([1.0, 0.0, 0.0, 0.0, 0.0])) # Methane only
fuel.SetTemperature(300.0) # K
fuel.SetPressure(101325.0) # Pa
fuel.SetMassFlow(1.0) # kg/s

air.SetOverallComposition(Array[float]([0.0, 0.21, 0.0, 0.0, 0.79])) # Air composition (O2 and N2)
air.SetTemperature(300.0) # K
air.SetPressure(101325.0) # Pa
air.SetMassFlow(5.0) # kg/s

'air: mass flow set to 5 kg/s'

In [10]:
# property package
Thermo_Package = sim.CreateAndAddPropertyPackage("Peng-Robinson (PR)")

In [11]:
# Creating the conversion reaction for CH4 combustion
from System.Collections.Generic import Dictionary
from System import String, Double

# Stoichiometric coefficients (positive for products, negative for reactants)
stoich = Dictionary[String, Double]()
stoich.Add("Methane", -1.0)
stoich.Add("Oxygen", -2.0)
stoich.Add("CarbonDioxide", 1.0)
stoich.Add("Water", 2.0)

# Base compound (the one for which conversion is defined)
base_compound = "Methane"

# Reaction phase – "Vapor" because combustion is gaseous
reaction_phase = "Vapor"

# Conversion expression – here a constant 95%
# You can also use a function of temperature T (in K), e.g. "0.9 * exp(-1000/T)"
conversion_expr = "95"

# Create the reaction using the flowsheet's helper method
reaction = sim.CreateConversionReaction(
    "Methane Combustion",           # name
    "Complete combustion of methane",  # description
    stoich,                          # stoichiometry dictionary
    base_compound,                   # base compound
    reaction_phase,                  # phase
    conversion_expr                   # conversion expression
)
# Add the reaction to the flowsheet
sim.AddReaction(reaction)

# ------------------------------------------------------------
# 3. Create a reaction set and add the reaction to it
# ------------------------------------------------------------
reaction_set = sim.CreateReactionSet("Combustion Set", "Contains methane combustion")
sim.AddReactionSet(reaction_set)

# Add the reaction to the set (enabled = True, rank = 0)
sim.AddReactionToSet(reaction.Name, reaction_set.Name, True, 0)

# Setting reaction set to the conversion reactor
combustor.ReactorOperationMode.Adiabatic
combustor.DeltaP = 0
combustor.ReactionSetID = reaction_set.Name
combustor.ReactionSet = reaction_set

In [12]:
# request a calculation
errors = interf.CalculateFlowsheet4(sim)
list(errors)

[]

In [13]:
# save file

fileNameToSave = Path.Combine(Environment.GetFolderPath(Environment.SpecialFolder.Desktop), "/workspace/00 FlowSheet Automation/12 Conversion Reactor Automation/12 Conversion Reactor Automation.dwxmz")

interf.SaveFlowsheet(sim, fileNameToSave, True)